In [26]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.basemap import Basemap
from matplotlib.colors import ListedColormap, TwoSlopeNorm
import cartopy.crs as ccrs
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter, LatitudeLocator
import cartopy.feature as cfeature
from numpy import meshgrid
from typing import List
import bottleneck as bn
import os
from scipy.stats import percentileofscore as pos
from multiprocessing import Pool
from scipy.signal import detrend
import matplotlib.colors as mcolors
from scipy import stats
import pickle
import cftime
from datetime import datetime
import config.dataUtils as dutils
import config.plotUtils as putils
import config.metricUtils as mutils
import config.fdUtils_1 as fdutils_1
import config.fdUtils_2 as fdutils_2
import config.fdUtils_3 as fdutils_3
import config.fdUtils_4 as fdutils_4
import config.detrendUtils as trendUtils
import config.percentUtils as pctUtils
import config.sesrUtils as sUtils
import config.deltasesrUtils as dsesrUtils
import config.trendplotUtils as tpUtils
import config.FDtimeSeriesPlot as fdplot
import config.fdStatsPlot as fdStats
import config.FDcompareUtils as fdCompare
import config.realtimeNLDAS as realtime
import config.fdREALTIME as fdCurrent
import config.STATIC as call


'''NEXT STEP IS TO DETREND ESR BASED ON THE DOY CLIMATOLOGY ALREADY CREATED'''
'''NEXT STEP IS TO DO FLASH DROUGHT STEPS 1-4'''

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


'NEXT STEP IS TO DO FLASH DROUGHT STEPS 1-4'

In [ ]:
year_ranges_tuple_1 = call.long_clim
year_ranges_tuple_2 = call.short_clim
window_of_centered_mean=call.mean_rolling_length


In [ ]:
#and for RZSM
RZSM_detrend_climatology = save_RZSM_trend_slope_climatology_for_detrending_REALTIME(rzsm, climatology)
RZSM_detrend  = RZSM_detrended(rzsm,RZSM_detrend_climatology,climatology)
RZSM_percentile =  REALTIME_percentile_RZSM_by_day_of_year(window,climatology,RZSM_detrend, rzsm)

In [ ]:
# TODO:

# 1.) convert RZSM to m3/m3 (by dividing by 1000)
# 2.) calculate 7-day centered rolling mean
dutils.convert_RZSM_and_take_centered_mean_and_save(window_of_centered_mean, year_ranges_tuple_1, year_ranges_tuple_2,recompute=True)
# 3.) de-trend RZSM
trendUtils.detrend_RZSM_by_doy() 
# 4.) calculate percentiles (based on all dates and also by_doy)
[pctUtils.make_percentile_from_RZSM_by_day_of_year(window,year_ranges_tuple_1,year_ranges_tuple_2) for window in np.arange(1)]
[pctUtils.make_percentile_from_RZSM_from_all_dates(window,year_ranges_tuple_1,year_ranges_tuple_2) for window in np.arange(1)]
# 5.) Then add a caveat to FD identification in which the RZSM percentile is below the 30th percentile

In [ ]:
# '''re-write this code in the morning'''

# % Identifies flash droughts using SESR and delta_SESR
# % flash_droughts_identification:
# % 1) identifies flash droughts using four criteria
# % 2) saves flash drought information into flash_drought_directory

# % Add paths
# addpath('../../../Functions')

# % Reshape sesr and standardized_sesr_change into 3 dimensions
# % (space,pentads,years)
# [i,t] = size(standardized_sesr_change);
# pentads_in_year = t/years;
# sesr_3d = reshape(sesr,[i,pentads_in_year,years]);
# standardized_sesr_change_3d_all =...
#     reshape(standardized_sesr_change,[i,pentads_in_year,years]);
# clear i t

# % % Detrend sesr and standardized_sesr_change
# % for g = 1:size(sesr_3d,1)
# %     for p = 1:size(sesr_3d,2)
# %         sesr_3d(g,p,:) = detrend(permute(sesr_3d(g,p,:),[3 1 2]));
# % %         standardized_sesr_change_3d(g,p,:) =...
# % %             detrend(permute(standardized_sesr_change_3d(g,p,:),[3 1 2]));
# %     end
# %     disp(g/size(sesr_3d,1)*100)
# % end
# % clear g p

# % Reshape detrended data into 2 dimensions (space,time)
# clear sesr standardized_sesr_change
# [i,~,~] = size(sesr_3d);
# sesr = reshape(sesr_3d,[i,pentads_in_year*years]);
# standardized_sesr_change = reshape(standardized_sesr_change_3d_all,...
#     [i,pentads_in_year*years]);
# clear pentads_in_year i

# % Find percentiles from sesr and standardized_sesr_change
# sesr_drought_prctile = 20;
# sesr_change_prctile = 25;
# drought_sesr_all = prctile(sesr_3d,sesr_drought_prctile,3);
# min_sesr_change_all =...
#     prctile(standardized_sesr_change_3d_all,sesr_change_prctile,3);
# clear sesr_3d sesr_drought_prctile sesr_change_prctile

# % Prep variables for parfor
# sesr_all = sesr;
# clear sesr
# standardized_sesr_change_all = standardized_sesr_change;
# clear standardized_sesr_change

# % Create flash drought directory to store information of flash droughts
# flash_drought_directory_all = cell(size(sesr_all,1),1);

# total_pixels = size(sesr_all,1);

# % Find flash drought events
# disp('Finding flash drought events')
# % count = 0; n_ind = 1; total_grids = size(sesr,1);
# % Loop through each spatial point
# i_segs = 1:10000:total_pixels;
# i_all = 1:total_pixels;
# i_segs(end + 1) = total_pixels + 1;
# for i_seg = 1:(length(i_segs) - 1)
#     i_range = i_segs(i_seg):(i_segs(i_seg + 1) - 1);
#     lat_seg = lat(i_range);
#     sesr = sesr_all(i_range,:);
#     standardized_sesr_change = standardized_sesr_change_all(i_range,:);
#     standardized_sesr_change_3d = standardized_sesr_change_3d_all(i_range,:,:);
#     drought_sesr = drought_sesr_all(i_range,:);
#     min_sesr_change = min_sesr_change_all(i_range,:);
#     flash_drought_directory = repmat({struct('fd_start',[],'fd_end',[],...
#         'fd_length',[],'fd_intensity',[],...
#         'not_in_grow_seas',zeros(years,1),...
#         'not_in_drought',zeros(years,1),...
#         'not_long_enough',zeros(years,1),...
#         'not_fast_enough',zeros(years,1))},length(i_range),1);
#     parfor i = 1:length(i_range)
#         continuous_flash = 0;
#         was_flash = 0;
#         length_flash_drought = 0;
#         % Loop through all pentads
#         for n = 1:(n_pentads - 1)
#             n_y = mod(n,73); if n_y == 0; n_y = 73; end
#             if continuous_flash ~= 1
#                if (standardized_sesr_change(i,n) < min_sesr_change(i,n_y))
#                    continuous_flash = 1;
#                    length_flash_drought = length_flash_drought + 1;
#                end
#             elseif continuous_flash == 1
#                 if  standardized_sesr_change(i,n) < min_sesr_change(i,n_y)
#                     continuous_flash = 1;
#                     length_flash_drought = length_flash_drought + 1;
#                 else
#                     continuous_flash = 0;
#                     was_flash = 1;
#                 end
#             end
#             if was_flash == 1
#                 % Check if flash drought is in growing season
#                 length_flash_drought_adj = length_flash_drought + 1;
#                 fd_n_begin = n - length_flash_drought;
#                 fd_n_end = n;
#                 in_grow_seas = 0;
#                 if lat_seg(i) > 30
#                     if (any(pentad_idx_in_growing_season_nh == (n - 1)) ||...
#                             any(pentad_idx_in_growing_season_nh == fd_n_begin))
#                         in_grow_seas = 1;
#                     else
#                         in_grow_seas = 0;
#                     end
#                 elseif lat_seg(i) < -30
#                     if (any(pentad_idx_in_growing_season_sh == (n - 1)) ||...
#                             any(pentad_idx_in_growing_season_sh == fd_n_begin))
#                         in_grow_seas = 1;
#                     else
#                         in_grow_seas = 0;
#                     end
#                 elseif lat_seg(i) <= 30 && lat_seg(i) >= -30
#                     in_grow_seas = 1;
#                 end
                
#                 % Store missing criteria data
#                 y_idx = ceil(n/73);
#                 if in_grow_seas == 0
#                     flash_drought_directory{i}.not_in_grow_seas(y_idx) =...
#                         flash_drought_directory{i}.not_in_grow_seas(y_idx) +...
#                         1;
#                 end
#                 if sesr(i,n) >= drought_sesr(i,n_y)
#                     flash_drought_directory{i}.not_in_drought(y_idx) =...
#                         flash_drought_directory{i}.not_in_drought(y_idx) +...
#                         1;
#                 end
#                 if length_flash_drought < 5
#                     flash_drought_directory{i}.not_long_enough(y_idx) =...
#                         flash_drought_directory{i}.not_long_enough(y_idx) +...
#                         1;
#                 end

#                 % Check if flash drought ended in drought and is minimum length
#                 if (sesr(i,n) < drought_sesr(i,n_y)) &&...
#                         (length_flash_drought >= 5) && (in_grow_seas == 1)
#                     mean_sesr_change = mean(standardized_sesr_change(i,...
#                         (fd_n_begin):(fd_n_end - 1))); '''Add a -1 in my script'''

#                     n_y_begin = mod(fd_n_begin,73);
#                     if n_y_begin == 0; n_y_begin = 73; end
#                     n_y_end = mod(fd_n_end,73);
#                     if n_y_end == 0; n_y_end = 73; end
#                     if n_y_end == 1; n_y_end = 74; end
#                     if n_y_end - n_y_begin > 0
#                         sesr_change_distribution =...
#                             standardized_sesr_change_3d(i,n_y_begin:(n_y_end - 1),:);
#                     else
#                         sesr_change_distribution =...
#                             standardized_sesr_change_3d(i,n_y_begin:73,:);
#                         sesr_change_distribution =...
#                             [sesr_change_distribution,...
#                             standardized_sesr_change_3d(i,1:(n_y_end - 1),:)];
#                     end
#                     min_mean_sesr_change =...
#                         prctile(sesr_change_distribution(:),[25,15,7.5,2.5]); % [25,20,15,10] [30,25,20,15] [25,15,7.5,2.5]
# %                     clear sesr_change_distribution
                    
#                     % Store missing criteria data
#                     if mean_sesr_change >= min_mean_sesr_change(1)
#                     flash_drought_directory{i}.not_fast_enough(y_idx) =...
#                         flash_drought_directory{i}.not_fast_enough(y_idx) +...
#                         1;
#                     end
                    
#                     % Check to see if event rapidly developed toward drought
#                     if mean_sesr_change < min_mean_sesr_change(1)
# %                         sesr_n_begin = n - length_flash_drought_adj - 12 - 1;
# %                         if sesr_n_begin < 1; sesr_n_begin = 1; end
# %                         sesr_n_end = n - length_flash_drought_adj - 1;
# %                         pre_mean_sesr = mean(sesr(i,sesr_n_begin:sesr_n_end),'omitnan');

#                         % Determine flash drought intensity from mean_sesr_change
#                         fd_intensity = -1;
#                         if mean_sesr_change < min_mean_sesr_change(4)
#                             fd_intensity = 4;
#                         elseif mean_sesr_change < min_mean_sesr_change(3)
#                             fd_intensity = 3;
#                         elseif mean_sesr_change < min_mean_sesr_change(2)
#                             fd_intensity = 2;
#                         elseif mean_sesr_change < min_mean_sesr_change(1)
#                             fd_intensity = 1;
#                         end

#                         % Store data into flash_drought_directory
#                         flash_drought_directory{i}.fd_start =...
#                             [flash_drought_directory{i}.fd_start;...
#                             pentad2date_v2(fd_n_begin,base_year)];
#                         flash_drought_directory{i}.fd_end =...
#                             [flash_drought_directory{i}.fd_end;...
#                             pentad2date_v2(fd_n_end,base_year)];
# %                         flash_drought_directory{i}.mean_sesr_change =...
# %                             [flash_drought_directory{i}.mean_sesr_change;...
# %                             mean_sesr_change];
#                         flash_drought_directory{i}.fd_length =...
#                             [flash_drought_directory{i}.fd_length;...
#                             length_flash_drought_adj];
#                         flash_drought_directory{i}.fd_intensity =...
#                             [flash_drought_directory{i}.fd_intensity;...
#                             fd_intensity];
# %                         flash_drought_directory{i}.pre_sesr =...
# %                             [flash_drought_directory{i}.pre_sesr;...
# %                             pre_mean_sesr];
# %                         if was_interrupted == 0
# %                             flash_drought_directory{i}.non_moderated_fd =...
# %                                 [flash_drought_directory{i}.non_moderated_fd;...
# %                                 1];
# %                         else
# %                             flash_drought_directory{i}.non_moderated_fd =...
# %                                 [flash_drought_directory{i}.non_moderated_fd;...
# %                                 0];
# %                         end
#                     end
#                 end
#                 was_flash = 0;
#                 length_flash_drought = 0;
# %                 clear min_mean_sesr_change
#             end
#         end

# %         if n_ind > 1
# %             fprintf(1,repmat('\b',1,count));
# %             count = fprintf('%3.2f%% complete',n_ind/total_grids*100);
# %         end
# %         n_ind = n_ind + 1;
#     end
#     flash_drought_directory_all(i_range) = flash_drought_directory(:);
#     disp(i_seg/(length(i_segs) - 1)*100)
# end
# delete(gcp('nocreate'))
# clear count n_ind total_grids i new_flash continuous_flash was_flash...
#     was_interrupted flash_interruption length_flash_drought n n_y...
#     standardized_sesr_change standardized_sesr_change_3d fd_n_begin...
#     fd_n_end mean_sesr_change n_y_begin n_y_end min_mean_sesr_change...
#     drought_sesr pentad_idx_in_growing_season_nh pentad_idx_in_growing_season_sh...
#     sesr_n_begin sesr_n_end in_grow_seas lat lon length_flash_drought_adj...
#     fd_intensity pre_mean_sesr min_sesr_change n_pentads sesr base_year...
#     length_flash_drought_new
# disp(' ')

# % Restore all arrays back to original lat/lon array size
# flash_drought_directory_original = cell(prod(lat_lon_original_size),1);
# flash_drought_directory_original(isin_land) = flash_drought_directory_all;
# flash_drought_directory =...
#     reshape(flash_drought_directory_original,lat_lon_original_size);
# clear lat_lon_original_size flash_drought_directory_original isin_conus...
#     isin_land

# % Save flash drought directory information
# if exist(fn_name,'file') && save_fd_dir
#         delete(fn_name)
# end
# save(fn_name,'flash_drought_directory','-v7.3')
# clearvars -except ds dataset

# Load Data

In [ ]:
'''
obs = original observation files (after converting to mm/day) for both EVP and PEVPR
yearly_sum = EVP and PEVPR the annual summation of EVP and PEVPR.
obs_mean = 7-day centered rolling mean of EVP, PEVPR, and ESR
monthly_sum = Take monthly sum of each month
'''

'''Select the year ranges to make the distribution over for SESR and all other functions. These are years.'''

max_distribution_window_range=5 #days +/- 
num_processes = min(max_distribution_window_range+1,20)

lat = 15
lon = 70

# dutils.detrend_data_and_take_centered_mean_and_save(window_of_centered_mean=7, year_ranges_tuple_1=year_ranges_tuple_1,
#                                                    year_ranges_tuple_2=year_ranges_tuple_2)

# dutils.save_data_and_take_centered_mean(window_of_centered_mean=7, year_ranges_tuple_1=year_ranges_tuple_1,
#                                                    year_ranges_tuple_2=year_ranges_tuple_2)

dutils.load_ET_PET_RefET_and_compute_ESR_baseline_years(window_of_centered_mean, year_ranges_tuple_1, year_ranges_tuple_2,recompute=True)

In [ ]:
a=xr.open_dataset(f'{call.noah_dir}/esr_rolling_mean_0.50_degrees.nc')
a


In [ ]:
lat = 15
lon = 70

#plot location 
a=xr.open_dataset('Data/Noah/esr_rolling_mean_0.50_degrees.nc')
location = a['EVP']
for Y in range(location.lat.shape[0]):
    for X in range(location.lon.shape[0]):
        if Y==lat and X==lon:
            pass
        else:
            location[:,Y,X] = np.nan

# location[300,:,:].plot()

# Plotting a specific slice (time index 300)
fig = plt.figure(figsize=(10, 6))
ax = plt.subplot(1, 1, 1, projection=ccrs.PlateCarree())

# Plot data slice
location[300, :, :].plot(ax=ax, transform=ccrs.PlateCarree(), cmap='viridis', vmin=0, vmax=1)

# Add map features
ax.add_feature(cfeature.BORDERS, linestyle='-', linewidth=1)
ax.add_feature(cfeature.COASTLINE, linewidth=1)
ax.add_feature(cfeature.RIVERS, linestyle='-', linewidth=0.5, edgecolor='blue')
ax.add_feature(cfeature.LAKES, alpha=0.5, edgecolor='blue')
ax.add_feature(cfeature.LAND, facecolor='none')
ax.add_feature(cfeature.OCEAN)
# ax.add_feature(cfeature.MOUNTAINS, facecolor='gray', alpha=0.3)  # Adding mountains feature


# Set plot title and labels
ax.set_title('Data at time index 300 with US Border', fontsize=14)
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)

In [ ]:
a['ESR_refet'][:,lat,lon].plot()
print('test')
a['ESR_pet'][:,lat,lon].plot()

# Plots

## 1.) year to year variation in ET/PET before de-trending to see what is going on with the last 10 years values being all high in some grid cells

## 2.) kde plots

## 3.) yearly and monthly statistics


In [ ]:
obs = xr.open_dataset(f'{call.noah_dir}/esr_rolling_mean_0.50_degrees.nc').load()

In [ ]:
putils.make_ESR_plots(year_ranges_tuple_1,year_ranges_tuple_2)


# Remove linear trend

In [ ]:
'''Remove the linear trend from ESR
Cannot do the xarray ufunc because of missing values. 

We have already applied an interpolate np.nan function, masked values less than 0 with 0. 
Values greather than 3 are given np.nan. Then we interpolate the np.nan values.'''
trendUtils.detrend_ESR_by_doy(recompute=True) 
trendUtils.detrend_RZSM_by_doy() 


# Plot trend values for different statistics

In [ ]:
#Save the trend values based on different conditions. Saved the data for each doy into Data/Noah/doy_trend
trendUtils.compute_trends_by_year_and_decade_and_20_years(year_ranges_tuple=call.long_clim)
tpUtils.plot_change_per_year_by_season()
tpUtils.plot_change_per_two_decades_by_season()
tpUtils.plot_change_per_1_decade_by_season()
# def plot_trend_stats()

In [ ]:
trendUtils.RZSM_trends_by_year_and_decade_and_20_years(year_ranges_tuple=call.long_clim)

In [ ]:
a=xr.open_dataset(f'{call.noah_dir}/ESR_de-trend_years_2000-2019.nc')
print('ESR_PET')
a['ESR_pet'][:,lat,lon].plot()

In [ ]:
print('ESR_PET detrend')
a['ESR_pet_detrend'][:,lat,lon].plot()

In [ ]:
print('ESR_refet')
a['ESR_refet'][:,lat,lon].plot()

In [ ]:
print('ESR_refet detrend')
a['ESR_refet_detrend'][:,lat,lon].plot()

# Make the mean and standard deviation for different windows to compute the Standardized Evaporative Stress Ratio (SESR)

In [ ]:
'''
Create the mean and standard deviation by window range. See the function for more information.

But we are saving this information rather than directly applying the function for simplicity.
'''
[mutils.create_ESR_mean_std(window,year_ranges_tuple_1,year_ranges_tuple_2) for window in np.arange(1)]

# a=xr.open_dataset('Data/ESR_mean_std/ESR_mean_std_window_size_0_years_1981-2020.nc')
# a['mean'][:,40,30].plot()

# Use the mean and standard deviation to standardize the ESR

In [ ]:
'''Compute only SESR which is the (value - mu / std) based on the window size'''
[sUtils.create_SESR_by_doy(window,year_ranges_tuple_1,year_ranges_tuple_2) for window in np.arange(1)]
a=xr.open_dataset('Data/Noah/SESR/SESR_window_size_0_years_1981-2020.nc')
a['SESR_pet'][:,lat,lon].plot()
# a['SESR'][1400,:,:].plot()

In [ ]:
a['SESR_refet'][:,lat,lon].plot()

# Make delta SESR

In [ ]:
[dsesrUtils.delta_SESR_return_std_and_mean_diff_across_years(window,year_ranges_tuple_1,year_ranges_tuple_2) for window in np.arange(1)]


a=xr.open_dataset('Data/Noah/dzSESR_standardized/dzSESR_window_size_0_years_1981-2020.nc')
a['dzSESR_refet'][:,lat,lon].plot()

In [ ]:
a['dzSESR_pet'][:,lat,lon].plot()

# Make dzSESR average month across all years plots

In [ ]:
# putils.plot_dzSESR_monthly_average_statistics_of_average_value(window=0)

# Next make percentiles based on SESR_detrend and delta_SESR_detrend

In [ ]:

[pctUtils.make_percentile_from_SESR_and_dzSESR_by_day_of_year(window,year_ranges_tuple_1,year_ranges_tuple_2) for window in np.arange(1)]
[pctUtils.make_percentile_from_SESR_and_dzSESR_from_all_dates(window,year_ranges_tuple_1,year_ranges_tuple_2) for window in np.arange(1)]


In [ ]:
a=xr.open_dataset('Data/Noah/dzSESR_percentile/dzSESR_percentile_by_doy_window_size_0_years_1981-2020.nc')
a['dzSESR_pct_refet'][:,lat,lon].plot()

In [ ]:
b=xr.open_dataset('Data/Noah/dzSESR_percentile/dzSESR_percentile_all_dates_window_size_0_years_1981-2020.nc')
b['dzSESR_pct_refet'][:,lat,lon].plot()

In [ ]:
a['dzSESR_pct_refet'][:,lat,lon].plot()

In [ ]:
a['SESR_pct_refet'][:,lat,lon].plot()

In [ ]:
a['SESR_pct_refet'][:,lat,lon].plot()

# Plot standardized SESR for 2012 May, June, July for the different time periods

In [ ]:
'''Create the case studies for the 2012 Central US flash drought just to ensure that there are reasonable values'''

for pet_or_refet in ['pet','refet']:
    [putils.plot_dzSESR_values_for_case_studies(window=window, year_ranges_tuple=year_ranges_tuple_1, start_date='2012-05-01', num_weeks=12, pet_or_refet = pet_or_refet) for window in range(1)]
    [putils.plot_dzSESR_values_for_case_studies(window=window, year_ranges_tuple=year_ranges_tuple_2, start_date='2012-05-01', num_weeks=12, pet_or_refet = pet_or_refet) for window in range(1)]


# Flash Drought Identification

In [ ]:
'''
Steps 1-3 are handled in Step 1 function
1.) First criteria, must last at least 4 weeks in length (28 days)

2.) Second criteria, final SESR value must be below the 20th percentile

3a.) delta_SESR must be at or below the 40th percentile between individual weeks

3b.) No more than one delta_SESR above the 40th percentile foloowing a delta_SESR that meets criterion 3a

#########################################################################################################################
Step4 is handled in Step 2 function
4) The mean change in SESR (or delta SESR) during the entire length of the flash drought must be less than the 25th percentile.

Percentiles for criteria 2 and 3 were taken from the distribution of SESR and ΔSESR at each grid point and
specific pentads for all years used from the data set, while percentiles for criterion 4 were taken from the
distribution of ΔSESR at each grid point for pentads that were encompassed within the flash drought event
for all years used from the data set.

This code is copied verbatim from Christian et al. but.... he did not provide all the code for some functions which leaves it up to some 
subjective interpretation. But we will see how it goes. I will do the drought categories later if time allows.

Source: https://zenodo.org/records/7796371

'''

[fdutils_1.FD_step_1(window,year_ranges_tuple_1, year_ranges_tuple_2,all_days_or_only_doy_percentile = 'all_dates', day_of_week_to_analyze_FD = 'Wednesday') for window in np.arange(1)]
[fdutils_1.FD_step_1(window,year_ranges_tuple_1, year_ranges_tuple_2,all_days_or_only_doy_percentile = 'by_doy', day_of_week_to_analyze_FD = 'Wednesday') for window in np.arange(1)]


'''By selecting only a single day of the week, we avoid any potential re-writes of the data do to looping. This simplifies everything
because GEFSv12 reforecasts are only on Wednesdays anyways.
'''



In [ ]:
#All dates only 
a=xr.open_dataset('Data/Noah/dzSESR_FD_step_1_from_all_dates_percentile/dzSESR_fd_step_1_from_all_dates_percentile_window_size_0_years_1981-2020.nc')
b=xr.open_dataset('Data/Noah/dzSESR_FD_step_1_from_all_dates_percentile/dzSESR_fd_step_1_from_all_dates_percentile_window_size_0_years_2000-2019.nc')

def print_number_of_FD_weeks(file1, file2, name_of_percentile_dist):
    for file in [file1,file2]:
        dates = pd.to_datetime(file.time.values)
        for pet_or_refet in ['pet','refet']:
            print(f"{pet_or_refet} {name_of_percentile_dist} :Number of FD weeks across all time and grid cells for {pet_or_refet} years {dates[0].year}-{dates[-1].year} is {np.count_nonzero(file[f'fd_{pet_or_refet}'].values==1)}")
print_number_of_FD_weeks(a, b, 'from_all_dates_percentile' )

In [ ]:
#All dates only 
a=xr.open_dataset('Data/Noah/dzSESR_FD_step_1_from_by_doy_percentile/dzSESR_fd_step_1_from_by_doy_percentile_window_size_0_years_1981-2020.nc')
b=xr.open_dataset('Data/Noah/dzSESR_FD_step_1_from_by_doy_percentile/dzSESR_fd_step_1_from_by_doy_percentile_window_size_0_years_2000-2019.nc')

def print_number_of_FD_weeks(file1, file2, name_of_percentile_dist):
    for file in [file1,file2]:
        dates = pd.to_datetime(file.time.values)
        for pet_or_refet in ['pet','refet']:
            print(f"{pet_or_refet} {name_of_percentile_dist} :Number of FD weeks across all time and grid cells for {pet_or_refet} years {dates[0].year}-{dates[-1].year} is {np.count_nonzero(file[f'fd_{pet_or_refet}'].values==1)}")
print_number_of_FD_weeks(a, b, 'from_by_doy_percentile' )

In [ ]:

[fdutils_2.FD_step_2(window,year_ranges_tuple_1, year_ranges_tuple_2,all_dates_or_only_doy_percentile = 'all_dates', ) for window in np.arange(1)]
[fdutils_2.FD_step_2(window,year_ranges_tuple_1, year_ranges_tuple_2,all_dates_or_only_doy_percentile = 'by_doy', ) for window in np.arange(1)]


In [ ]:
#All dates only 
a=xr.open_dataset('Data/Noah/dzSESR_FD_step_2_from_all_dates_percentile/dzSESR_fd_step_2_from_all_dates_percentile_window_size_0_years_1981-2020.nc')
b=xr.open_dataset('Data/Noah/dzSESR_FD_step_2_from_all_dates_percentile/dzSESR_fd_step_2_from_all_dates_percentile_window_size_0_years_2000-2019.nc')

def print_number_of_FD_weeks(file1, file2, name_of_percentile_dist):

    for file in [file1,file2]:
        for s2_or_s3 in ['s2','s3']:
            if s2_or_s3 == 's2':
                step_ = 'find within crop growing season (not winter)'
            elif s2_or_s3 == 's3':
                step_ = 'find if mean dsSESR percentile is less than 25th'
        
            dates = pd.to_datetime(file.time.values)
            for pet_or_refet in ['pet','refet']:
                print(f"{pet_or_refet} \n{name_of_percentile_dist}\n{step_}\nNumber of FD weeks across all time and grid cells for {pet_or_refet} years {dates[0].year}-{dates[-1].year} is \n{np.count_nonzero(file[f'fd_{pet_or_refet}_{s2_or_s3}'].values==1)}\n")
                
print_number_of_FD_weeks(a, b, 'from_all_dates_percentile' )


In [ ]:
# Now remove FD events if they are longer than 24 weeks (indicating long-term drought)
#And also remove if they are not within the growing season

[fdutils_3.FD_step_3(window,year_ranges_tuple_1, year_ranges_tuple_2,all_dates_or_only_doy_percentile = 'all_dates', num_weeks_FD_to_longterm_drought=24) for window in np.arange(1)]
[fdutils_3.FD_step_3(window,year_ranges_tuple_1, year_ranges_tuple_2,all_dates_or_only_doy_percentile = 'by_doy', num_weeks_FD_to_longterm_drought=24) for window in np.arange(1)]


In [ ]:
#Now find if they are also below the 30th percentile.

[fdutils_4.FD_step_4(window,year_ranges_tuple_1, year_ranges_tuple_2,all_dates_or_only_doy_percentile = 'all_dates', min_RZSM_percentile = 30) for window in np.arange(1)]

[fdutils_4.FD_step_4(window,year_ranges_tuple_1, year_ranges_tuple_2,all_dates_or_only_doy_percentile = 'by_doy', min_RZSM_percentile = 30) for window in np.arange(1)]



# Case studies for after FD classification

In [ ]:
'''Create the case studies for the 2012 Central US flash drought just to ensure that there are reasonable values'''

for s1_or_s2_or_s3 in ['s1','s2','s3','s4','s5']:
    for all_dates_or_only_doy_percentile in ['all_dates','by_doy']:
        for pet_or_refet in ['pet','refet']:
            [putils.plot_binary_FD_values_for_case_studies(window=window, year_ranges_tuple=year_ranges_tuple_1, start_date='2012-05-01', num_weeks=12, pet_or_refet = pet_or_refet, all_dates_or_only_doy_percentile=all_dates_or_only_doy_percentile, s1_or_s2_or_s3 = s1_or_s2_or_s3) for window in range(1)]
            [putils.plot_binary_FD_values_for_case_studies(window=window, year_ranges_tuple=year_ranges_tuple_2, start_date='2012-05-01', num_weeks=12, pet_or_refet = pet_or_refet, all_dates_or_only_doy_percentile=all_dates_or_only_doy_percentile, s1_or_s2_or_s3 = s1_or_s2_or_s3) for window in range(1)]


# Plot FD statistics for different drought categories and percent of years

In [ ]:
# for s1_or_s2_or_s3 in ['s1','s2','s3','s4','s5']:
for s1_or_s2_or_s3 in ['s2','s3','s4','s5']:
    for all_dates_or_only_doy_percentile in ['all_dates','by_doy']:
        for pet_or_refet in ['pet','refet']:
            fdStats.plot_percent_of_years_with_FD(0,year_ranges_tuple_1,year_ranges_tuple_2,all_dates_or_only_doy_percentile,s1_or_s2_or_s3,pet_or_refet)

In [ ]:
'''Currently, the percent of years plots do not look correct which means that something is wrong with the methodology.

To fix this, the only thing that I am uncertain about is step 4 - The mean change in SESR (or delta SESR) during the entire 
length of the flash drought must be less than the 25th percentile.

This could imply:
1.) That after all the flash droughts have been classified, then we re-calculate a percentile distribution based on ONLY the deltaSESR
values. This doens't make much sense because we are re-percentiling a distribution which was already percentiled. But technically it works bec

2.) The current way of my interpretation is that we simply average the value of the deltaSESR across the current FD time step

'''


# Now find the agreement between different FD steps and between pet and refet and determine if RZSM is important

In [ ]:
for all_dates_or_only_doy_percentile in ['all_dates','by_doy']:
    compute_FD_similarity_statistics(window, year_ranges_tuple_1, year_ranges_tuple_2,all_dates_or_only_doy_percentile, )

# Making realtime NLDAS (up to August 2024)

In [3]:

#For testing
# year_ranges_tuple_1=(1981,2020)
# year_ranges_tuple_2=(2000,2019)
# window=0
# pet_or_refet = 'pet'
# all_dates_or_doy='by_doy'
# climatology = year_ranges_tuple_1




Standardizing from climatology (1981, 2020) for variable ESR_pet over range 1980-01-01T00:00:00.000000000 - 2024-07-29T00:00:00.000000000


/glade/work/klesinger/sesr/config/realtimeNLDAS.py:408: RuntimeWarning: All-NaN slice encountered
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)-np.nanmin(vals,axis=0))
/glade/work/klesinger/sesr/config/realtimeNLDAS.py:408: RuntimeWarning: All-NaN slice encountered
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)-np.nanmin(vals,axis=0))
/glade/work/klesinger/sesr/config/realtimeNLDAS.py:408: RuntimeWarning: All-NaN slice encountered
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)-np.nanmin(vals,axis=0))
/glade/work/klesinger/sesr/config/realtimeNLDAS.py:408: RuntimeWarning: All-NaN slice encountered
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)-np.nanmin(vals,axis=0))
/glade/work/klesinger/sesr/config/realtimeNLDAS.py:408: RuntimeWarning: All-NaN slice encountered
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)-np.nanmin(vals,axis

Standardizing from climatology (1981, 2020) for variable ESR_refet over range 1980-01-01T00:00:00.000000000 - 2024-07-29T00:00:00.000000000


/glade/work/klesinger/sesr/config/realtimeNLDAS.py:408: RuntimeWarning: All-NaN slice encountered
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)-np.nanmin(vals,axis=0))
/glade/work/klesinger/sesr/config/realtimeNLDAS.py:408: RuntimeWarning: invalid value encountered in divide
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)-np.nanmin(vals,axis=0))
/glade/work/klesinger/sesr/config/realtimeNLDAS.py:408: RuntimeWarning: All-NaN slice encountered
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)-np.nanmin(vals,axis=0))
/glade/work/klesinger/sesr/config/realtimeNLDAS.py:408: RuntimeWarning: invalid value encountered in divide
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)-np.nanmin(vals,axis=0))
/glade/work/klesinger/sesr/config/realtimeNLDAS.py:408: RuntimeWarning: All-NaN slice encountered
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)

Standardizing from (1981, 2020) for variable ESR_pet_dtrnd range 1980-01-01T00:00:00.000000000 - 2024-07-29T00:00:00.000000000


...

Standardizing from (1981, 2020) for variable ESR_refet_dtrnd range 1980-01-01T00:00:00.000000000 - 2024-07-29T00:00:00.000000000
Computing the weekly difference for var SESR_pet_dtrnd across all years and subtracting the already computed climatology for (1981, 2020).


/glade/work/klesinger/sesr/config/realtimeNLDAS.py:223: RuntimeWarning: invalid value encountered in subtract
  weekly_difference = selected_data.values - selected_data_previous.values


Computing the weekly difference for var SESR_refet_dtrnd across all years and subtracting the already computed climatology for (1981, 2020).
Standardizing from climatology (1981, 2020) for variable RZSM over range 1979-01-02T00:00:00.000000000 - 2024-07-30T00:00:00.000000000


/glade/work/klesinger/sesr/config/realtimeNLDAS.py:502: RuntimeWarning: All-NaN slice encountered
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)-np.nanmin(vals,axis=0))
/glade/work/klesinger/sesr/config/realtimeNLDAS.py:502: RuntimeWarning: All-NaN slice encountered
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)-np.nanmin(vals,axis=0))
/glade/work/klesinger/sesr/config/realtimeNLDAS.py:502: RuntimeWarning: All-NaN slice encountered
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)-np.nanmin(vals,axis=0))
/glade/work/klesinger/sesr/config/realtimeNLDAS.py:502: RuntimeWarning: All-NaN slice encountered
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)-np.nanmin(vals,axis=0))
/glade/work/klesinger/sesr/config/realtimeNLDAS.py:502: RuntimeWarning: All-NaN slice encountered
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)-np.nanmin(vals,axis

File '/glade/work/klesinger/sesr/Data/Noah/dzSESR_percentile_REALTIME/RZSM_pct_by_doy_window_size0_clim1981-2020.nc' has been removed.
Finding RZSM_pct_dtrnd percentile of score by day of year for each grid cell window_size_0_climatology_(1981, 2020).


In [64]:
#Now add steps for flash drought identification
def all_FD_steps_with_RZSM(recompute=True):
    esr, rzsm, clim_esr, clim_rzsm, trend_esr, trend_rzsm, clim_sesr, ESR_detrend, \
    dzSESR, dzSESR_percentile, ESR_detrend_climatology,RZSM_detrend_climatology,  RZSM_detrend, \
    RZSM_percentile, climatology, window, all_dates_or_doy = realtime.load_data_before_FD_classification_REALTIME(climatology=call.long_clim, 
                                                                                                                  window=call.window, 
                                                                                                                  all_dates_or_doy=call.all_dates_or_doy, 
                                                                                                                  recompute_ESR_trend_slope=False,
                                                                                                                 recompute_dzSESR_percentiles=False,
                                                                                                                 recompute_RZSM_percentiles=False)
        
    fd_s1 = fdCurrent.FD_step_1_REALTIME(dzSESR_percentile, climatology, window, all_dates_or_doy, RZSM_percentile, 
                                         day_of_week_to_analyze_FD = call.init_day_of_week, 
                                         recompute =recompute)
    
    
    fd_s1 = fdCurrent.FD_step_2_REALTIME(fd_s1, climatology, window, all_dates_or_doy, recompute=recompute )
    fd_s1 = fdCurrent.FD_step_3_REALTIME(fd_s1, climatology, window, all_dates_or_doy,recompute=recompute)
    fd_s1 = fdCurrent.FD_step_4_REALTIME(fd_s1, RZSM_percentile, climatology, window, all_dates_or_doy,recompute=recompute)
    return 'Completed Realtime NLDAS flash drought classification for all steps'

all_FD_steps_with_RZSM(recompute=True)

Standardizing from climatology (1981, 2020) for variable ESR_pet over range 1980-01-01T00:00:00.000000000 - 2024-07-29T00:00:00.000000000


/glade/work/klesinger/sesr/config/realtimeNLDAS.py:408: RuntimeWarning: All-NaN slice encountered
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)-np.nanmin(vals,axis=0))
/glade/work/klesinger/sesr/config/realtimeNLDAS.py:408: RuntimeWarning: All-NaN slice encountered
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)-np.nanmin(vals,axis=0))
/glade/work/klesinger/sesr/config/realtimeNLDAS.py:408: RuntimeWarning: All-NaN slice encountered
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)-np.nanmin(vals,axis=0))
/glade/work/klesinger/sesr/config/realtimeNLDAS.py:408: RuntimeWarning: All-NaN slice encountered
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)-np.nanmin(vals,axis=0))
/glade/work/klesinger/sesr/config/realtimeNLDAS.py:408: RuntimeWarning: All-NaN slice encountered
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)-np.nanmin(vals,axis

Standardizing from climatology (1981, 2020) for variable ESR_refet over range 1980-01-01T00:00:00.000000000 - 2024-07-29T00:00:00.000000000


/glade/work/klesinger/sesr/config/realtimeNLDAS.py:408: RuntimeWarning: All-NaN slice encountered
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)-np.nanmin(vals,axis=0))
/glade/work/klesinger/sesr/config/realtimeNLDAS.py:408: RuntimeWarning: invalid value encountered in divide
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)-np.nanmin(vals,axis=0))
/glade/work/klesinger/sesr/config/realtimeNLDAS.py:408: RuntimeWarning: All-NaN slice encountered
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)-np.nanmin(vals,axis=0))
/glade/work/klesinger/sesr/config/realtimeNLDAS.py:408: RuntimeWarning: invalid value encountered in divide
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)-np.nanmin(vals,axis=0))
/glade/work/klesinger/sesr/config/realtimeNLDAS.py:408: RuntimeWarning: All-NaN slice encountered
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)

Standardizing from (1981, 2020) for variable ESR_pet_dtrnd range 1980-01-01T00:00:00.000000000 - 2024-07-29T00:00:00.000000000


...

/glade/work/klesinger/sesr/config/realtimeNLDAS.py:502: RuntimeWarning: All-NaN slice encountered
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)-np.nanmin(vals,axis=0))
/glade/work/klesinger/sesr/config/realtimeNLDAS.py:502: RuntimeWarning: All-NaN slice encountered
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)-np.nanmin(vals,axis=0))
/glade/work/klesinger/sesr/config/realtimeNLDAS.py:502: RuntimeWarning: All-NaN slice encountered
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)-np.nanmin(vals,axis=0))
/glade/work/klesinger/sesr/config/realtimeNLDAS.py:502: RuntimeWarning: All-NaN slice encountered
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)-np.nanmin(vals,axis=0))
/glade/work/klesinger/sesr/config/realtimeNLDAS.py:502: RuntimeWarning: All-NaN slice encountered
  min_max_standardize = (vals- np.nanmin(vals,axis=0))/(np.nanmax(vals,axis=0)-np.nanmin(vals,axis

Removed /glade/work/klesinger/sesr/Data/Noah/dzSESR_FD_step_1_REALTIME_from_by_doy_percentile/dzSESR_fd_step_1_from_by_doy_percentile_window0_clim1981-2020_|_2021-01-01_thru_2024-07-30.nc.

Working on fd_1_pet.
Using percentiles from by_doy.
Window size 0.
Dates_2021-01-01_thru_2024-07-30.

Checking if length >= 4 weeks,
SESR less than 20th percentile,
dzSESR less than 40th percentile from climatology_1981-2020.
#############################################################


Working on fd_1_refet.
Using percentiles from by_doy.
Window size 0.
Dates_2021-01-01_thru_2024-07-30.

Checking if length >= 4 weeks,
SESR less than 20th percentile,
dzSESR less than 40th percentile from climatology_1981-2020.
#############################################################

Loading combined growing season crop file.
Removed /glade/work/klesinger/sesr/Data/Noah/dzSESR_FD_step_2_REALTIME_from_by_doy_percentile/dzSESR_fd_step_2_from_by_doy_percentile_window0_clim1981-2020_|_2021-01-06_thru_2024-07-24.n

/glade/work/klesinger/sesr/config/fdREALTIME.py:438: RuntimeWarning: Mean of empty slice
  fd_s3[f'fd_5_{pet_or_refet}'][potential_fd[0]:potential_fd[-1],Y,X] = 0


Working on refet and finding if the RZSM percentile is lower than 30 percentile.
Window size 0.
climatology_1981-2020.


'Completed Realtime NLDAS flash drought classification for all steps'